# IRP Stage 3 — Experiment Tracking
## Scenario 2: A single data scientist participating in an ML competition
**MLOps Group Project | Section 1, Group 5**  
*Maria-Irina Popa, Enzo Jerez, Roberto Cummings, Jia Yi Rachel Lee, Thomas Christian Matenco*

---

need to run this in terminal
mlflow server \
  --backend-store-uri sqlite:///mlflow.db \
  --default-artifact-root ./mlartifacts \
  --host 127.0.0.1 \
  --port 5001

In [ ]:
import mlflow
import mlflow.sklearn
import sklearn
import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("inventory-risk-stage3-team")

<Experiment: artifact_location=('/Users/ljyr/Documents/IE/Term 2/1 MLOPS Machine Learning '
 'Operations/IRP-Section1Group5/03-experiment-tracking/mlruns/1'), creation_time=1774312385929, experiment_id='1', last_update_time=1774312385929, lifecycle_stage='active', name='inventory-risk-stage3', tags={}, workspace='default'>

In [10]:
train = pd.read_parquet("../02-features-modelling/data/train.parquet")
val   = pd.read_parquet("../02-features-modelling/data/val.parquet")
test  = pd.read_parquet("../02-features-modelling/data/test.parquet")

In [11]:
print(train.shape, val.shape, test.shape)
print(train.head(2))

(53900, 30) (12300, 30) (6100, 30)
        Date Store ID Product ID     Category Region  Inventory Level  \
0 2022-01-08     S001      P0001    Furniture   East              231   
1 2022-01-09     S001      P0001  Electronics  South              373   

   Units Sold  Units Ordered  Demand Forecast  Price  ...  Inventory_Change  \
0           2            119             0.84  66.30  ...              -1.0   
1         350            178           352.24  41.72  ...             117.0   

  Inventory_Change_Pct  Days_of_Stock  Inventory_vs_Rolling7 Sales_Velocity  \
0            -0.001965     254.000000             173.857143       0.003937   
1             0.230315       1.785714             251.285714       0.560000   

   Coverage_Ratio  Forecast_Error  Order_to_Inventory  Risk_Label_Current  \
0      508.000000            1.16            0.234252           Safe Zone   
1        1.774358           -2.24            0.284800       Stockout Risk   

      Risk_Label  
0  Stockout Risk  

In [12]:
TARGET = "Risk_Label"

drop_cols = [
    "Risk_Label",           # target
    "Risk_Label_Current",   # leakage
    "Store ID",             # ID
    "Product ID",           # ID
    "Date",                 # not needed for now
    "Demand Forecast",      # used in label
    "Demand_Forecast_Clean" # used in label
]

X_train = train.drop(columns=drop_cols).copy()
X_val   = val.drop(columns=drop_cols).copy()
X_test  = test.drop(columns=drop_cols).copy()

y_train_raw = train[TARGET].copy()
y_val_raw   = val[TARGET].copy()
y_test_raw  = test[TARGET].copy()

numerical_features = [
    "Inventory Level",
    "Units Sold",
    "Units Ordered",
    "Price",
    "Discount",
    "Units_Sold_Lag1",
    "Inventory_Change_Pct",
    "Days_of_Stock",
    "Sales_Velocity",
    "Coverage_Ratio",
    "Forecast_Error",
    "Order_to_Inventory"
]

categorical_features = [
    "Category",
    "Region",
    "Weather Condition",
    "Seasonality"
]

# Keep exactly the Stage 2 feature subset
X_train = X_train[numerical_features + categorical_features].copy()
X_val   = X_val[numerical_features + categorical_features].copy()
X_test  = X_test[numerical_features + categorical_features].copy()

print(X_train.shape, X_val.shape, X_test.shape)
print("Numerical:", numerical_features)
print("Categorical:", categorical_features)


(53900, 16) (12300, 16) (6100, 16)
Numerical: ['Inventory Level', 'Units Sold', 'Units Ordered', 'Price', 'Discount', 'Units_Sold_Lag1', 'Inventory_Change_Pct', 'Days_of_Stock', 'Sales_Velocity', 'Coverage_Ratio', 'Forecast_Error', 'Order_to_Inventory']
Categorical: ['Category', 'Region', 'Weather Condition', 'Seasonality']


In [13]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train = le.fit_transform(y_train_raw)
y_val   = le.transform(y_val_raw)
y_test  = le.transform(y_test_raw)

class_names = list(le.classes_)
print(class_names)

['Overstock Risk', 'Safe Zone', 'Stockout Risk']


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)


In [15]:
# --- Logistic Regression pipeline (aligned with Stage 2) ---
preprocessor_logit = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
])

logit_pipeline = ImbPipeline([
    ("preprocessor", preprocessor_logit),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

# Ensure correct dtypes for SMOTENC in tree models
for col in categorical_features:
    X_train[col] = X_train[col].astype("object")
    X_val[col] = X_val[col].astype("object")
    X_test[col] = X_test[col].astype("object")

cat_indices = [X_train.columns.get_loc(col) for col in categorical_features]
smote_nc = SMOTENC(categorical_features=cat_indices, random_state=42)

preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)


In [16]:
# --- Random Forest pipeline (aligned with Stage 2) ---
rf_pipeline = ImbPipeline(steps=[
    ("smote", smote_nc),
    ("preprocessor", preprocessor_tree),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=10,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ))
])

# --- XGBoost pipeline (aligned with Stage 2) ---
xgb_pipeline = SkPipeline(steps=[
    ("preprocessor", preprocessor_tree),
    ("model", XGBClassifier(
        objective="multi:softmax",
        num_class=3,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="mlogloss"
    ))
])

classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, class_weights))
sample_weights = np.array([class_weight_dict[y] for y in y_train])

print("Class weights:", class_weight_dict)


Class weights: {np.int64(0): np.float64(4.549675023212628), np.int64(1): np.float64(0.454047679218263), np.int64(2): np.float64(1.73072600584401)}


In [17]:
models = [
    ("Logistic Regression", logit_pipeline),
    ("Random Forest", rf_pipeline),
    ("XGBoost", xgb_pipeline)
]

In [18]:
results = []

all_labels = list(range(len(class_names)))

for model_name, model in models:
    with mlflow.start_run(run_name=model_name) as run:
        mlflow.set_tag("model_type", model_name)
        mlflow.set_tag("run_time", datetime.datetime.now().isoformat())
        mlflow.set_tag("description", f"{model_name} on inventory risk data (aligned to Stage 2)")

        mlflow.log_param("sklearn_version", sklearn.__version__)
        mlflow.log_param("numerical_features", ", ".join(numerical_features))
        mlflow.log_param("categorical_features", ", ".join(categorical_features))
        mlflow.log_param("n_classes", len(np.unique(y_train)))

        # Fit exactly as in Stage 2
        if model_name == "XGBoost":
            model.fit(X_train, y_train, model__sample_weight=sample_weights)
        else:
            model.fit(X_train, y_train)

        split_data = {
            "train": (X_train, y_train),
            "val": (X_val, y_val),
            "test": (X_test, y_test),
        }

        split_metrics = {}

        for split_name, (X_split, y_split) in split_data.items():
            y_pred = model.predict(X_split)

            metrics_dict = {
                f"{split_name}_accuracy": accuracy_score(y_split, y_pred),
                f"{split_name}_precision_macro": precision_score(y_split, y_pred, average="macro", zero_division=0),
                f"{split_name}_recall_macro": recall_score(y_split, y_pred, average="macro", zero_division=0),
                f"{split_name}_f1_macro": f1_score(y_split, y_pred, average="macro"),
            }

            mlflow.log_metrics(metrics_dict)
            split_metrics.update(metrics_dict)

            if split_name == "val":
                cm = confusion_matrix(y_split, y_pred, labels=all_labels)
                cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
                cm_csv = f"confusion_matrix_{model_name.lower().replace(' ', '_')}.csv"
                cm_png = f"confusion_matrix_{model_name.lower().replace(' ', '_')}.png"
                report_txt = f"classification_report_{model_name.lower().replace(' ', '_')}.txt"

                cm_df.to_csv(cm_csv)
                mlflow.log_artifact(cm_csv)

                plt.figure(figsize=(6, 5))
                sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
                plt.title(f"Validation Confusion Matrix ({model_name})")
                plt.ylabel("True label")
                plt.xlabel("Predicted label")
                plt.tight_layout()
                plt.savefig(cm_png)
                plt.close()
                mlflow.log_artifact(cm_png)

                report = classification_report(
                    y_split,
                    y_pred,
                    labels=all_labels,
                    target_names=class_names,
                    zero_division=0,
                )
                with open(report_txt, "w") as f:
                    f.write(report)
                mlflow.log_artifact(report_txt)

        input_example = X_train.iloc[[0]]
        mlflow.sklearn.log_model(model, artifact_path="model", input_example=input_example)

        print(
            f"Logged run for {model_name} | "
            f"val_accuracy={split_metrics['val_accuracy']:.3f}, "
            f"val_precision_macro={split_metrics['val_precision_macro']:.3f}, "
            f"val_recall_macro={split_metrics['val_recall_macro']:.3f}, "
            f"val_f1_macro={split_metrics['val_f1_macro']:.3f}"
        )

        results.append({
            "model_name": model_name,
            "run_id": run.info.run_id,
            "train_accuracy": round(split_metrics["train_accuracy"], 4),
            "train_precision_macro": round(split_metrics["train_precision_macro"], 4),
            "train_recall_macro": round(split_metrics["train_recall_macro"], 4),
            "train_f1_macro": round(split_metrics["train_f1_macro"], 4),
            "val_accuracy": round(split_metrics["val_accuracy"], 4),
            "val_precision_macro": round(split_metrics["val_precision_macro"], 4),
            "val_recall_macro": round(split_metrics["val_recall_macro"], 4),
            "val_f1_macro": round(split_metrics["val_f1_macro"], 4),
            "test_accuracy": round(split_metrics["test_accuracy"], 4),
            "test_precision_macro": round(split_metrics["test_precision_macro"], 4),
            "test_recall_macro": round(split_metrics["test_recall_macro"], 4),
            "test_f1_macro": round(split_metrics["test_f1_macro"], 4),
        })


2026/03/24 02:51:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/24 02:51:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternat

Logged run for Logistic Regression | val_accuracy=0.326, val_precision_macro=0.332, val_recall_macro=0.332, val_f1_macro=0.268


2026/03/24 02:52:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/24 02:52:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternat

Logged run for Random Forest | val_accuracy=0.505, val_precision_macro=0.327, val_recall_macro=0.327, val_f1_macro=0.303


2026/03/24 02:52:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/24 02:52:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternat

Logged run for XGBoost | val_accuracy=0.444, val_precision_macro=0.333, val_recall_macro=0.331, val_f1_macro=0.311


In [19]:
results_df = pd.DataFrame(results).sort_values(
    ["val_f1_macro", "val_recall_macro", "val_precision_macro", "val_accuracy"],
    ascending=False
).reset_index(drop=True)

results_df


,model_name,run_id,train_accuracy,train_precision_macro,train_recall_macro,train_f1_macro,val_accuracy,val_precision_macro,val_recall_macro,val_f1_macro,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro
0,XGBoost,9aa981abd06444758607429313a5c3fd,0.6851,0.5914,0.7782,0.6178,0.4436,0.3333,0.3309,0.3110,0.4398,0.3338,0.3358,0.3140
1,Random Forest,45d1e5d089524b84a2d5100440cf3a17,0.5177,0.3672,0.3722,0.3363,0.5048,0.3268,0.3273,0.3034,0.4997,0.3402,0.3360,0.3097
2,Logistic Regression,e811ba93698c4707aeb2b8e5b9d21062,0.3291,0.3370,0.3428,0.2754,0.3259,0.3318,0.3322,0.2681,0.3246,0.3335,0.3378,0.2713
